# 01 Data Loading

## Purpose
Load the synthetic EHR source tables, confirm that the expected files are available, and document the basic shape of each table before cleaning or modeling.

This supports the course requirement to describe the dataset clearly before applying machine learning.

## Imports

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "models"
for directory in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, DATA_DIR / "processed"]:
    directory.mkdir(parents=True, exist_ok=True)

import pandas as pd

from src.data_cleaning import load_raw_data, find_data_file

## Data loading

In [2]:
raw = load_raw_data(DATA_DIR)
raw.keys()

dict_keys(['patients', 'diagnoses', 'lab_results', 'medications', 'outcomes'])

In [3]:
summary_rows = []
for name, df in raw.items():
    summary_rows.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "unique_patients": df["patient_id"].nunique() if "patient_id" in df.columns else None,
    })

summary = pd.DataFrame(summary_rows)
summary

,table,rows,columns,unique_patients
0,patients,100000,28,100000
1,diagnoses,274592,8,100000
2,lab_results,2827722,10,100000
3,medications,364174,10,94067
4,outcomes,11001,12,11001


In [4]:
for name, df in raw.items():
    print(f"\n{name.upper()} shape={df.shape}")
    display(df.head())


PATIENTS shape=(100000, 28)


,patient_id,age,sex,bmi,systolic_bp,diastolic_bp,heart_rate,temperature_f,smoking_status,alcohol_use,...,dx_heart_failure,dx_atrial_fibrillation,dx_chronic_kidney_disease,dx_copd,dx_asthma,dx_depression,dx_anxiety,dx_hypothyroidism,dx_osteoarthritis,dx_type1_diabetes
0,P0000001,66,M,23.5,148,81,64,98.4,former,light,...,0,0,1,0,0,0,0,0,1,0
1,P0000002,75,M,24.8,158,86,45,99.5,never,moderate,...,0,0,0,0,1,0,0,0,1,0
2,P0000003,82,M,17.8,135,57,90,98.2,never,light,...,0,0,1,0,0,0,0,0,0,0
3,P0000004,73,F,28.1,118,83,102,98.9,former,moderate,...,0,0,0,0,0,0,0,0,0,0
4,P0000005,86,F,30.6,156,81,56,98.7,never,heavy,...,0,0,0,0,1,1,0,1,0,0



DIAGNOSES shape=(274592, 8)


,patient_id,visit_date,visit_type,primary_diagnosis,primary_icd10,secondary_diagnoses,secondary_icd10s,provider_specialty
0,P0000001,2018-07-26,telehealth,hyperlipidemia,E78.5,chronic_kidney_disease,N18.3,internal_medicine
1,P0000001,2020-04-09,outpatient,hyperlipidemia,E78.5,chronic_kidney_disease|osteoarthritis,N18.3|M19.90,nephrology
2,P0000001,2023-08-12,emergency,chronic_kidney_disease,N18.3,hyperlipidemia,E78.5,internal_medicine
3,P0000001,2022-11-04,outpatient,osteoarthritis,M19.90,hyperlipidemia|chronic_kidney_disease,E78.5|N18.3,internal_medicine
4,P0000002,2020-07-29,outpatient,osteoarthritis,M19.90,type2_diabetes,E11.9,nephrology



LAB_RESULTS shape=(2827722, 10)


,patient_id,test_date,test_name,value,unit,reference_low,reference_high,flag,is_abnormal,delta_from_normal
0,P0000001,2019-03-21,WBC,9.551,K/uL,4.5,11.0,normal,0,1.801
1,P0000001,2019-03-21,total_cholesterol,225.729,mg/dL,100.0,199.0,high,1,76.229
2,P0000001,2019-03-21,HDL,67.620,mg/dL,40.0,100.0,normal,0,-2.380
3,P0000001,2019-03-21,potassium,4.423,mEq/L,3.5,5.0,normal,0,0.173
4,P0000001,2019-03-21,sodium,142.463,mEq/L,136.0,145.0,normal,0,1.963



MEDICATIONS shape=(364174, 10)


,patient_id,medication,dose,unit,frequency,indication,start_date,duration_days,is_generic,adherence_pct
0,P0000001,Rosuvastatin,10.0,mg,daily,hyperlipidemia,2020-03-06,30,1,84.5
1,P0000001,Lisinopril,10.0,mg,daily,chronic_kidney_disease,2020-01-22,30,1,81.0
2,P0000001,Erythropoietin,4000.0,units,weekly,chronic_kidney_disease,2021-05-24,90,0,77.1
3,P0000001,Ibuprofen,400.0,mg,three times daily,osteoarthritis,2020-03-16,180,1,90.2
4,P0000002,Metoprolol,25.0,mg,twice daily,hypertension,2018-12-21,365,1,100.0



OUTCOMES shape=(11001, 12)


,patient_id,admission_date,discharge_date,length_of_stay_days,icu_admission,icu_days,in_hospital_death,discharge_disposition,readmitted_30d,days_to_readmission,primary_drg,total_charges_usd
0,P0000002,2024-08-06,2024-08-09,3,0,0,0,home_health,0,NaN,291,8589.79
1,P0000003,2022-05-06,2022-05-08,2,0,0,0,hospice,1,28.0,392,22628.17
2,P0000011,2021-12-30,2022-01-08,9,0,0,0,rehab,0,NaN,392,27565.97
3,P0000014,2022-06-30,2022-07-04,4,0,0,0,home,0,NaN,194,10332.33
4,P0000040,2020-09-19,2020-09-20,1,0,0,0,rehab,0,NaN,470,2389.61


## Date coverage

In [5]:
date_checks = {
    "diagnoses": ["visit_date"],
    "lab_results": ["test_date"],
    "medications": ["start_date"],
    "outcomes": ["admission_date", "discharge_date"],
}

rows = []
for table, columns in date_checks.items():
    for column in columns:
        dates = pd.to_datetime(raw[table][column], errors="coerce")
        rows.append({
            "table": table,
            "date_column": column,
            "min_date": dates.min(),
            "max_date": dates.max(),
            "missing_dates": dates.isna().sum(),
        })

date_summary = pd.DataFrame(rows)
date_summary

,table,date_column,min_date,max_date,missing_dates
0,diagnoses,visit_date,2018-01-01,2024-12-30,0
1,lab_results,test_date,2018-01-01,2024-12-30,0
2,medications,start_date,2018-01-01,2021-07-01,0
3,outcomes,admission_date,2018-01-01,2024-12-30,0
4,outcomes,discharge_date,2018-01-03,2025-01-10,0


## Key findings

- The project uses five patient-level EHR tables: patients, diagnoses, lab results, medications, and hospital outcomes.
- Dated clinical events cover 2018 through 2024, which allows time-aware feature engineering.
- The notebooks will avoid leakage by using a cutoff date for predictors and future dates for targets.

## Saved outputs

In [6]:
summary.to_csv(TABLES_DIR / "01_raw_table_summary.csv", index=False)
date_summary.to_csv(TABLES_DIR / "01_date_coverage_summary.csv", index=False)
print("Saved table summaries to", TABLES_DIR)

Saved table summaries to /Users/chinmaynaringrekar/PythonProject/Disease Progression and Multimorbidity Prediction/outputs/tables
